In [ ]:
#| default_exp lisette_client

# Stateful Lisette chat (`lisette_client`)

This module is named **`lisette_client`** (not `lisette`) so a script run from inside `string_therapy/` does not shadow the PyPI **`lisette`** package on `sys.path`.

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional, Sequence

import os
import sys


def _unshadow_pypi_lisette() -> None:
    """If ``lisette.py`` sits in the process cwd, it shadows the PyPI ``lisette`` package."""
    root = os.getcwd()
    if not sys.path:
        return
    head = sys.path[0]
    if head in ("", root) and os.path.isfile(os.path.join(root, "lisette.py")):
        if not os.path.isdir(os.path.join(root, "lisette")):
            sys.path.pop(0)


_unshadow_pypi_lisette()
from lisette.core import Chat

import string_therapy.therapy_config as tc


_chats is a module level cache ( a single dict shared by every import of this module ) .
* keys: (sid, variant) -- a 2-tuple of strings: session id and chat
* values: Lisette ``Chat`` objects (typed as ``object`` here because the type checker may not know the exact Chat class without importing lisette.core at module load time

This ensures one chat instance per (session, variant) for the lifetime of the process, so repeated calls reuse the same object and conversation state(history) is preserved instead of creating a new chat every time . 
It starts as {} and is filled when _get_chat creates a new ``chat`` for a key that isn't present yet 

In [ ]:
#| export
_chats: Dict[tuple[str, str], object] = {}


def _load_dotenv() -> None:
    tc.load_environment()

def _sync_claude_api_key() -> None:
    k = os.getenv("CLAUDE_API_KEY")

def _model() -> str :
    _load_dotenv() 
    if os.getenv("CLAUDE_API_KEY"):
        return "claude-sonnet-4-5-20250929"

def _get_chat(
    sid : str, 
    *, 
    variant: str = "default",
    tools: Optional[Sequence[Any]] = None, 
    sp: str = "",
):
    _load_dotenv()
    _sync_claude_api_key()
    key = (sid, variant)
    if key not in _chats:
        kw: dict = {}
        if tools:
            kw["tools"] = list(tools)
        if sp:
            kw["sp"] = sp 
        _chats[key] = Chat(_model(), **kw)
    return _chats[key]


def _cts(r: Any) -> str:
    """
    Assistant text from a LiteLLM / OpenAI-shaped completion
    """
    mc = r.choices[0].message.content
    if isinstance(mc, list):
        part: list[str] = []
        for block in mc:
            if isinstance(block, dict) and block.get("type") == "text":
                parts.append(str(block.get("text", "")))
        joined = "".join(parts)
        return joined if joined else str(mc)
    return str(mc) if mc is not None else ""
    
    
def lisette_chat(
    sid: str, 
    msg: str, 
    *, 
    max_tokens: int, 
    variant: str = "default", 
    tools: Optional[List[Any]] = None, 
    sp: str = "",
    max_steps: int = 2, 
    ) -> str :
    try:
        chat = _get_chat(sid, variant=variant, tools=tools, sp=sp)
    except ImportError:
        return ("Lisette is not installed")

    r = chat(msg.strip(), max_tokens=max_tokens, max_steps=max_steps)
    try : 
        return _cts(r)
    except (AttributeError, IndexError, TypeError):
        return str(r)
    
